# 06 - Train a perception encoder and test imagery transfer

This notebook orchestrates the leakage-safe baseline documented in `docs/brain_encoder_baseline.md`: core NSD image → frozen multilayer ViT features → PCA → voxelwise ridge → held-out core-NSD validation → frozen evaluation on NSD-Imagery vision and imagery.

**Scientific guardrail:** NSD-Imagery is never used to select layers, PCA dimension, ridge penalty, voxels, or training duration. Repeated core-NSD presentations are averaged before image-level splitting.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
DATA_ROOT = Path.home() / 'data' / 'nsd'  # edit if needed
SUBJECT = 1
MODEL_ID = 'facebook/dinov2-small'
METHOD = 'DINOv2_PyramidRidge'
WORK = REPO_ROOT / 'outputs' / '06_core_nsd_encoder' / f'subj{SUBJECT:02d}' / 'dinov2_small'
WORK.mkdir(parents=True, exist_ok=True)

print('Python:', sys.executable)
print('Data root:', DATA_ROOT)
print('Work directory:', WORK)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Python: /srv/conda/envs/nsdimagery/bin/python
Data root: /home/jovyan/data/nsd
Work directory: /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small
CUDA: True
Tesla T4


## 1. Estimate and download core NSD

Run the estimate before downloading. A five-session download is useful only as a pipeline smoke test; use all completed sessions for scientific results. Data must live on persistent storage.

In [2]:
RUN_ESTIMATE = False
RUN_DOWNLOAD = False
download_command = [
    'bash', str(REPO_ROOT / 'scripts' / 'download_core_nsd_encoder_mvp.sh'),
    '--subject', f'{SUBJECT:02d}',
    '--sessions', 'all',
    '--dest', str(DATA_ROOT),
]
print(' '.join(download_command + ['--estimate']))
if RUN_ESTIMATE:
    subprocess.run(download_command + ['--estimate'], check=True)
if RUN_DOWNLOAD:
    subprocess.run(download_command, check=True)

bash /home/jovyan/NHprojectNSDimagery/scripts/download_core_nsd_encoder_mvp.sh --subject 01 --sessions all --dest /home/jovyan/data/nsd --estimate


## 2. Prepare unique-image beta targets

The preparation reads one beta session at a time, masks `nsdgeneral`, Z-scores every voxel within session, averages repeated presentations of each image, and reserves the shared 1,000 NSD images for the final perception test.

In [3]:
RUN_PREPARE = False
prepare_command = [
    sys.executable, str(REPO_ROOT / 'scripts' / 'prepare_core_nsd_encoder_data.py'),
    '--data-root', str(DATA_ROOT),
    '--subject', str(SUBJECT),
    '--output-dir', str(WORK),
    '--split-mode', 'shared1000',
]
print(' '.join(prepare_command))
if RUN_PREPARE:
    subprocess.run(prepare_command, check=True)

/srv/conda/envs/nsdimagery/bin/python /home/jovyan/NHprojectNSDimagery/scripts/prepare_core_nsd_encoder_data.py --data-root /home/jovyan/data/nsd --subject 1 --output-dir /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small --split-mode shared1000


In [4]:
core_manifest_path = WORK / f'core_subj{SUBJECT:02d}_manifest.csv'
if core_manifest_path.is_file():
    core_manifest = pd.read_csv(core_manifest_path)
    display(core_manifest.groupby('split').agg(
        unique_images=('row_id', 'size'),
        mean_repeats=('n_repeats', 'mean'),
    ))
else:
    print('Run the preparation step first.')

Run the preparation step first.


## 3. Cache frozen multilayer image features

The default extracts DINOv2 hidden states 3, 6, 9, and 12. CLS plus 1×1 and 2×2 pooled patch features preserve both semantic and coarse spatial information. The pretrained backbone remains frozen.

In [5]:
RUN_CORE_FEATURES = False
core_features_path = WORK / 'core_features.npz'
feature_command = [
    sys.executable, str(REPO_ROOT / 'scripts' / 'extract_image_features.py'),
    '--manifest', str(core_manifest_path),
    '--nsd-stimuli', str(DATA_ROOT / 'nsddata_stimuli/stimuli/nsd/nsd_stimuli.hdf5'),
    '--model-id', MODEL_ID,
    '--layers', '3,6,9,12',
    '--pyramid-levels', '1,2',
    '--batch-size', '64',
    '--device', 'auto',
    '--output', str(core_features_path),
]
print(' '.join(feature_command))
if RUN_CORE_FEATURES:
    subprocess.run(feature_command, check=True)

/srv/conda/envs/nsdimagery/bin/python /home/jovyan/NHprojectNSDimagery/scripts/extract_image_features.py --manifest /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/core_subj01_manifest.csv --nsd-stimuli /home/jovyan/data/nsd/nsddata_stimuli/stimuli/nsd/nsd_stimuli.hdf5 --model-id facebook/dinov2-small --layers 3,6,9,12 --pyramid-levels 1,2 --batch-size 64 --device auto --output /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/core_features.npz


## 4. Select ridge penalty and test held-out perception

PCA and alpha selection see only the core-NSD train/validation identities. The selected model is refit on train+validation and evaluated once on the shared-1000 test images. A meaningful imagery analysis requires positive held-out perception predictivity.

In [6]:
RUN_FIT = False
encoder_path = WORK / 'encoder_model.npz'
summary_path = WORK / 'core_test_summary.json'
fit_command = [
    sys.executable, str(REPO_ROOT / 'scripts' / 'fit_ridge_encoder.py'),
    '--manifest', str(core_manifest_path),
    '--betas', str(WORK / f'core_subj{SUBJECT:02d}_betas.npy'),
    '--coordinates', str(WORK / f'core_subj{SUBJECT:02d}_coordinates.npy'),
    '--voxel-regions', str(WORK / f'core_subj{SUBJECT:02d}_voxel_regions.csv'),
    '--features', str(core_features_path),
    '--pca-components', '512',
    '--alphas', '0.1,1,10,100,1000',
    '--device', 'auto',
    '--method', METHOD,
    '--output-model', str(encoder_path),
    '--output-summary', str(summary_path),
    '--output-voxel-metrics', str(WORK / 'core_test_voxels.csv'),
]
print(' '.join(fit_command))
if RUN_FIT:
    subprocess.run(fit_command, check=True)
if summary_path.is_file():
    display(pd.json_normalize(json.loads(summary_path.read_text())))

/srv/conda/envs/nsdimagery/bin/python /home/jovyan/NHprojectNSDimagery/scripts/fit_ridge_encoder.py --manifest /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/core_subj01_manifest.csv --betas /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/core_subj01_betas.npy --coordinates /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/core_subj01_coordinates.npy --voxel-regions /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/core_subj01_voxel_regions.csv --features /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/core_features.npz --pca-components 512 --alphas 0.1,1,10,100,1000 --device auto --method DINOv2_PyramidRidge --output-model /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/encoder_model.npz --output-summary /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/core_test

## 5. Extract A/B target features

These are ground-truth targets, so this stage isolates encoder transfer without reconstruction error. Use exactly the same feature configuration as training.

In [7]:
RUN_TARGET_MANIFEST = False
RUN_TARGET_FEATURES = False
target_manifest_path = WORK / 'nsdimagery_AB_manifest.csv'
target_features_path = WORK / 'nsdimagery_AB_features.npz'
target_manifest_command = [
    sys.executable, str(REPO_ROOT / 'scripts' / 'make_nsdimagery_image_manifest.py'),
    '--data-root', str(DATA_ROOT),
    '--output', str(target_manifest_path),
]
target_feature_command = [
    sys.executable, str(REPO_ROOT / 'scripts' / 'extract_image_features.py'),
    '--manifest', str(target_manifest_path),
    '--model-id', MODEL_ID,
    '--layers', '3,6,9,12',
    '--pyramid-levels', '1,2',
    '--device', 'auto',
    '--output', str(target_features_path),
]
print(' '.join(target_manifest_command))
print(' '.join(target_feature_command))
if RUN_TARGET_MANIFEST:
    subprocess.run(target_manifest_command, check=True)
if RUN_TARGET_FEATURES:
    subprocess.run(target_feature_command, check=True)

/srv/conda/envs/nsdimagery/bin/python /home/jovyan/NHprojectNSDimagery/scripts/make_nsdimagery_image_manifest.py --data-root /home/jovyan/data/nsd --output /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/nsdimagery_AB_manifest.csv
/srv/conda/envs/nsdimagery/bin/python /home/jovyan/NHprojectNSDimagery/scripts/extract_image_features.py --manifest /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/nsdimagery_AB_manifest.csv --model-id facebook/dinov2-small --layers 3,6,9,12 --pyramid-levels 1,2 --device auto --output /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/nsdimagery_AB_features.npz


## 6. Frozen NSD-Imagery evaluation

The encoder predicts standardized voxel patterns from each target image. The evaluator correlates them across voxels with run-normalized, repeat-averaged vision and imagery betas in the paper's early, higher, and overall visual ROIs.

In [8]:
RUN_EVALUATION = False
evaluation_prefix = WORK / f'dinov2_subj{SUBJECT:02d}'
evaluation_command = [
    sys.executable, str(REPO_ROOT / 'scripts' / 'evaluate_encoder_nsdimagery.py'),
    '--data-root', str(DATA_ROOT),
    '--subject', str(SUBJECT),
    '--encoder-model', str(encoder_path),
    '--image-manifest', str(target_manifest_path),
    '--image-features', str(target_features_path),
    '--method', METHOD,
    '--output-prefix', str(evaluation_prefix),
]
print(' '.join(evaluation_command))
if RUN_EVALUATION:
    subprocess.run(evaluation_command, check=True)
evaluation_summary = Path(str(evaluation_prefix) + '_summary.csv')
if evaluation_summary.is_file():
    display(pd.read_csv(evaluation_summary))

/srv/conda/envs/nsdimagery/bin/python /home/jovyan/NHprojectNSDimagery/scripts/evaluate_encoder_nsdimagery.py --data-root /home/jovyan/data/nsd --subject 1 --encoder-model /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/encoder_model.npz --image-manifest /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/nsdimagery_AB_manifest.csv --image-features /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/nsdimagery_AB_features.npz --method DINOv2_PyramidRidge --output-prefix /home/jovyan/NHprojectNSDimagery/outputs/06_core_nsd_encoder/subj01/dinov2_small/dinov2_subj01


## Interpretation and next tier

1. Require positive held-out core-NSD voxel predictivity before interpreting transfer.
2. Compare imagery with the model's own vision score in each ROI; the claim concerns the size of the drop.
3. Repeat the full frozen analysis with CLIP for an explicitly semantic feature family.
4. Aggregate subjects and simple/real-scene targets with Notebook 05. Subjects remain the uncertainty unit.
5. Only after this baseline works, replace fixed spatial pooling with a learned factorized receptive-field readout and require improvement on the untouched core-NSD test set.